In [7]:
import ultralytics, datasets, huggingface_hub, pyarrow


In [ ]:
import os
os.makedirs("frames", exist_ok=True)
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \
  -o "input_video.mp4" \
  "https://www.youtube.com/watch?v=YcvECxtXoxQ"

[youtube] Extracting URL: https://www.youtube.com/watch?v=YcvECxtXoxQ
[youtube] YcvECxtXoxQ: Downloading webpage
[youtube] YcvECxtXoxQ: Downloading android vr player API JSON
[info] YcvECxtXoxQ: Downloading 1 format(s): 401+140
[download] Destination: input_video.f401.mp4
[download] 100% of    4.26GiB in 00:18:32 at 3.92MiB/s0;33m00:000mm
[download] Destination: input_video.f140.m4a
[download] 100% of   43.12MiB in 00:00:10 at 4.15MiB/s0;33m00:00
[Merger] Merging formats into "input_video.mp4"
Deleting original file input_video.f401.mp4 (pass -k to keep)
Deleting original file input_video.f140.m4a (pass -k to keep)


In [8]:
import os
os.makedirs("frames", exist_ok=True)
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \
  -o "input_video.mp4" \
  "https://www.youtube.com/watch?v=YcvECxtXoxQ"

[youtube] Extracting URL: https://www.youtube.com/watch?v=YcvECxtXoxQ
[youtube] YcvECxtXoxQ: Downloading webpage
[youtube] YcvECxtXoxQ: Downloading android vr player API JSON
[info] YcvECxtXoxQ: Downloading 1 format(s): 401+140
[download] Destination: input_video.f401.mp4
[download] 100% of    4.26GiB in 00:18:32 at 3.92MiB/s0;33m00:000mm
[download] Destination: input_video.f140.m4a
[download] 100% of   43.12MiB in 00:00:10 at 4.15MiB/s0;33m00:00
[Merger] Merging formats into "input_video.mp4"
Deleting original file input_video.f401.mp4 (pass -k to keep)
Deleting original file input_video.f140.m4a (pass -k to keep)


In [9]:
import os
os.makedirs("frames", exist_ok=True)
!ffmpeg -i input_video.mp4 -vf "fps=1/5" frames/frame_%04d.jpg -hide_banner -loglevel error
print(f"Frames extracted: {len(os.listdir('frames'))}")

Frames extracted: 559


In [ ]:
from ultralytics import YOLO
import pandas as pd
import os

model = YOLO("best.pt")  # seperately trained carparts model
print("Classes:", model.names)

frame_dir = "frames"
frame_files = sorted(os.listdir(frame_dir))
records = []

for i, fname in enumerate(frame_files):
    frame_path = os.path.join(frame_dir, fname)
    timestamp_sec = i * 5
    results = model(frame_path, verbose=False)
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
            records.append({
                "video_id": "YcvECxtXoxQ",
                "frame_index": i,
                "timestamp_sec": timestamp_sec,
                "filename": fname,
                "class_label": model.names[int(box.cls)],
                "confidence_score": float(box.conf),
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
            })

df = pd.DataFrame(records)
print(f"Total detections: {len(df)}")
df.head()

Classes: {0: 'back_bumper', 1: 'back_door', 2: 'back_glass', 3: 'back_left_door', 4: 'back_left_light', 5: 'back_light', 6: 'back_right_door', 7: 'back_right_light', 8: 'front_bumper', 9: 'front_door', 10: 'front_glass', 11: 'front_left_door', 12: 'front_left_light', 13: 'front_light', 14: 'front_right_door', 15: 'front_right_light', 16: 'hood', 17: 'left_mirror', 18: 'object', 19: 'right_mirror', 20: 'tailgate', 21: 'trunk', 22: 'wheel'}
Total detections: 1838


,video_id,frame_index,timestamp_sec,filename,class_label,confidence_score,x_min,y_min,x_max,y_max
0,YcvECxtXoxQ,0,0,frame_0001.jpg,wheel,0.914479,268.372559,1233.615356,2564.664795,2114.608154
1,YcvECxtXoxQ,0,0,frame_0001.jpg,front_glass,0.776871,1663.498169,554.115662,2765.039307,937.923401
2,YcvECxtXoxQ,0,0,frame_0001.jpg,front_right_door,0.745876,1118.003662,527.089050,1890.352417,1682.470947
3,YcvECxtXoxQ,0,0,frame_0001.jpg,front_bumper,0.706730,2475.484619,1069.883911,3745.479492,1777.992676
4,YcvECxtXoxQ,0,0,frame_0001.jpg,back_left_light,0.615593,148.823364,864.565613,257.793182,968.394836


In [17]:
df.to_parquet("detections.parquet", index=False)
print("Saved")

Saved


In [ ]:
from datasets import load_dataset
from PIL import Image
import io

# Load query images
ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")
print(f"Query images: {len(ds)}")

GAP_TOLERANCE = 10  

retrieval_results = []

for idx, sample in enumerate(ds):
    img = sample["image"]
    
    # Save temp image and run detector
    img.save("temp_query.jpg")
    results = model("temp_query.jpg", verbose=False)
    
    # Get detected classes from query
    query_classes = set()
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            query_classes.add(model.names[int(box.cls)])
    
    if not query_classes:
        continue

    # Match against video index
    for cls in query_classes:
        matches = df[df["class_label"] == cls].sort_values("timestamp_sec")
        if matches.empty:
            continue
        
        # Group into segments
        timestamps = matches["timestamp_sec"].tolist()
        segments = []
        start = timestamps[0]
        end = timestamps[0]
        
        for t in timestamps[1:]:
            if t - end <= GAP_TOLERANCE:
                end = t
            else:
                segments.append((start, end))
                start = t
                end = t
        segments.append((start, end))
        
        for start, end in segments:
            n = len(matches[(matches["timestamp_sec"] >= start) & (matches["timestamp_sec"] <= end)])
            retrieval_results.append({
                "query_index": idx,
                "query_timestamp": sample["timestamp"],
                "class_label": cls,
                "start_timestamp": start,
                "end_timestamp": end,
                "number_of_supporting_detections": n
            })

results_df = pd.DataFrame(retrieval_results)
print(f"Total retrieval results: {len(results_df)}")
results_df.head(10)

Generating train split: 100%|██████████| 65/65 [00:01<00:00, 35.97 examples/s]


Query images: 65
Total retrieval results: 8718


,query_index,query_timestamp,class_label,start_timestamp,end_timestamp,number_of_supporting_detections
0,0,00:00,right_mirror,0,15,3
1,0,00:00,right_mirror,50,50,1
2,0,00:00,right_mirror,185,185,1
3,0,00:00,right_mirror,1125,1245,48
4,0,00:00,right_mirror,1260,1270,5
5,0,00:00,right_mirror,1305,1305,1
6,0,00:00,right_mirror,1335,1340,3
7,0,00:00,right_mirror,1420,1420,1
8,0,00:00,right_mirror,1470,1470,1
9,0,00:00,right_mirror,1495,1620,53


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token="MY_TOKEN")

api.upload_file(
    path_or_fileobj="detections.parquet",
    path_in_repo="detections.parquet",
    repo_id="sania808/rav4-retrieval",
    repo_type="dataset"
)

print("Uploaded")

Processing Files (1 / 1): 100%|██████████| 83.0kB / 83.0kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded
